# ДЗ 2, опыт 16: скорость обучения — BPE-кеш и AMP FP16

**Короткий замер на T4, прежняя SwiGLU-модель 6.306M и train100k.** Цель — выбрать более быстрый цикл обучения для следующих опытов. Старые веса загружать не нужно; модель каждый раз начинает с одинаковых исходных весов.

| Режим | Подготовка данных | Вычисления |
| --- | --- | --- |
| A | BPE заново на каждом шаге, как в15 | FP32 |
| B | Заранее сохранённые BPE ID; прежние CPU tensors/padding/transfers | FP32 |
| C | Тот же BPE-кеш | AMP FP16 + GradScaler; веса и AdamW FP32 |

Три повтора. В каждом — одинаковые для A/B/C **100 шагов прогрева + 500 измеряемых попыток update**, batch8. Порядок режимов меняется: ABC/BCA/CAB. Отдельный короткий профиль20 шагов делается только в первом повторе. Максимум5460 попыток update; setup/BPE-кеш, прогрев, оценка и профиль не входят в основной таймер.

**Запуск:** новая Colab-сессия → GPU T4 → выполнить все ячейки по порядку. Затем скачать ZIP из последней ячейки и этот IPYNB с outputs. Если запуск прервался, последняя ячейка упакует уже сохранённые результаты. Перезапуск benchmark в том же каталоге заблокирован.

Это замер скорости и численной стабильности. Короткий validation loss — диагностика, не результат улучшения качества. Полный test1919 и долгое обучение здесь не запускаются. При полезном ускорении следующим отдельным опытом проверим качество на50k.

FP16 выбран для T4; BF16 не измеряется. Архитектура, BPE, данные, batch, dropout, AdamW, clipping и формула LR сохраняются. В AMP skipped updates не двигают LR; калибровочные skips прогрева записываются отдельно. Если есть skips в измеряемом блоке, режим не проходит контроль стабильности. Из-за FP16 траектории весов могут различаться.


In [ ]:
%pip install -q tokenizers==0.23.2 pyarrow


## 1. Настройки и сохранение результатов


In [ ]:
import base64
import csv
import gc
import hashlib
import io
import json
import math
import os
import random
import statistics
import time
import warnings
import zipfile
import zlib
from pathlib import Path
from urllib.request import urlretrieve

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import tokenizers
from tokenizers import Tokenizer
import pyarrow.parquet as pq
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence

SEED, TRAIN_SIZE, BATCH_SIZE = 42, 100000, 8
BENCH_WARMUP, MEASURE_STEPS, REPEATS, PROFILE_STEPS = 100, 500, 3, 20
D_MODEL, N_LAYERS, MAX_POSITIONS, SWIGLU_DIM = 256, 2, 2048, 680
PAD, BOS, EOS, TOKEN_OFFSET, TARGET_VOCAB = 0, 1, 2, 3, 4096
LEARNING_RATE, MIN_LEARNING_RATE = 0.001, 0.0001
WARMUP_STEPS, TOTAL_STEPS, SCHEDULE_TOTAL_STEPS = 1000, 100000, 50000
MODES = ('A_fp32_online_bpe', 'B_fp32_cached_bpe', 'C_amp_fp16_cached_bpe')
RUN_DIR = Path('hw02-training-speed-bpe-cache-amp-fp16')
RESULTS_FILE = RUN_DIR / 'results.json'
ARCHIVE_FILE = Path('hw02-training-speed-bpe4k-swiglu-t4-results.zip')
assert tokenizers.__version__ == '0.23.2', 'Выполни установку в новой сессии'
assert torch.cuda.is_available(), 'Включи GPU T4 в настройках Colab'
assert not RESULTS_FILE.exists(), 'Сначала скачай прежние результаты; используй новую сессию'
RUN_DIR.mkdir(exist_ok=True)
device = torch.device('cuda')

def sync():
    if device.type == 'cuda':
        torch.cuda.synchronize(device)

def sha(raw):
    return hashlib.sha256(raw).hexdigest()

def data_hash(rows):
    return sha(json.dumps(rows, ensure_ascii=False, sort_keys=True).encode())

def save_results():
    temporary = RESULTS_FILE.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(result, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(RESULTS_FILE)

result = {
    'experiment': '16-training-speed-bpe-cache-amp-fp16-swiglu-bpe4k-100k',
    'phase': 'setup', 'benchmark_started': False, 'benchmark_finished': False,
    'quality_experiment': False, 'test_evaluated': False,
    'total_attempted_updates': 0, 'total_successful_updates': 0,
    'runtime': {'device': str(device), 'gpu': torch.cuda.get_device_name(device),
                'compute_capability': list(torch.cuda.get_device_capability(device)),
                'torch': torch.__version__, 'cuda': torch.version.cuda,
                'tokenizers': tokenizers.__version__, 'cpu_threads': torch.get_num_threads(),
                'native_bf16_supported': torch.cuda.is_bf16_supported(including_emulation=False)},
    'config': {'seed': SEED, 'train_size': TRAIN_SIZE, 'batch': BATCH_SIZE,
               'warmup_attempts': BENCH_WARMUP, 'measured_attempts': MEASURE_STEPS,
               'repeats': REPEATS, 'profile_attempts_first_repeat': PROFILE_STEPS,
               'modes': list(MODES), 'lr': 'same warmup1000/cosine50000/floor as15',
               'adamw_weight_decay': 0.01, 'clip_norm': 1.0, 'dropout': 0.1,
               'data_preparation_factor': 'online BPE versus cached IDs; CPU tensors and padding unchanged'},
    'runs': [], 'profiles': [], 'fp32_pair_checks': [],
    'timer_scope': 'synchronized end-to-end train block: prepare/transfer/forward/loss/backward/unscale/clip/step/scaler/scheduler; excludes setup, warmup, logging, evaluation and separate serialized profile',
}
save_results()
print('RUNTIME', json.dumps(result['runtime']))


## 2. Прежние 100k train-пар и validation64


In [ ]:
revision = 'b2724689ff079e9c22efd548fe134bfb29036dbb'
data_dir = Path('hw02-data')
data_dir.mkdir(exist_ok=True)
train_file = data_dir / 'train.parquet'
data_started = time.perf_counter()
if not train_file.exists():
    urlretrieve(f'https://huggingface.co/datasets/SlavaYus/tinysms_qa_12m/resolve/{revision}/data/train/part-00000.parquet', train_file)
assert sha(train_file.read_bytes()) == '91831d582faa12d73b1a6b3878e496446cd5df074f17ee4ffea8cc2887c708ec'

def is_validation(question):
    key = hashlib.sha256(f'{SEED}:{question}'.encode()).digest()
    return int.from_bytes(key[:8], 'big') % 10 == 0

def eligible(row):
    return len(row['question'].encode()) <= 512 and len(row['code'].encode()) < 512

train, validation = [], None
for batch in pq.ParquetFile(train_file).iter_batches(batch_size=4096):
    rows = batch.to_pylist()
    if validation is None:
        validation = [r for r in rows if eligible(r) and is_validation(r['question'])][:64]
    for row in rows:
        if eligible(row) and not is_validation(row['question']):
            train.append(row)
            if len(train) == TRAIN_SIZE:
                break
    if len(train) == TRAIN_SIZE:
        break
assert len(train) == TRAIN_SIZE and len(validation) == 64
assert data_hash(train) == '7a141e91ce958206be7e18d3c50b5ccf72a10f1d94890c5f0c3927aa497e47b9'
assert data_hash(validation) == '0edf2b479ebdc103ec05efc704c7bb0ac8084e41a8a44b58afef224702563050'
assert not ({r['question'] for r in train} & {r['question'] for r in validation})
result['data'] = {'revision': revision, 'shard': 0, 'train_sha256': data_hash(train),
                  'validation64_sha256': data_hash(validation), 'train_examples': len(train),
                  'download_selection_seconds': time.perf_counter() - data_started}
save_results()
print('DATA', json.dumps(result['data']))


## 3. Прежний BPE08 и кеш ID
Кеширование измеряем отдельно от обучения. Источники, BOS/EOS, сдвиг labels и padding сохраняются.


In [ ]:
TOKENIZER_ZLIB_BASE64 = (
    '...'
)

tokenizer_raw = zlib.decompress(base64.b64decode(TOKENIZER_ZLIB_BASE64))
assert sha(tokenizer_raw) == '73614c921b43aa6ef715bb23bba583b96785a82304c1964f74ab11d8deef7f86'
tokenizer = Tokenizer.from_str(tokenizer_raw.decode('utf-8'))
VOCAB = tokenizer.get_vocab_size() + TOKEN_OFFSET
assert VOCAB == TARGET_VOCAB

def encode(text):
    return [i + TOKEN_OFFSET for i in tokenizer.encode(text, add_special_tokens=False).ids]

cache_started = time.perf_counter()
train_cache = [([BOS] + encode(r['question']) + [EOS], encode(r['code'])) for r in train]
validation_cache = [([BOS] + encode(r['question']) + [EOS], encode(r['code'])) for r in validation]
cache_seconds = time.perf_counter() - cache_started
assert all(len(s) <= MAX_POSITIONS and len(c) + 1 <= MAX_POSITIONS for s, c in train_cache + validation_cache)
assert all(TOKEN_OFFSET <= i < VOCAB for s, c in train_cache for i in s[1:-1] + c)
result['tokenization'] = {'sha256': sha(tokenizer_raw), 'vocab': VOCAB,
                          'training_performed': False, 'cache_seconds': cache_seconds,
                          'train_cached_pairs': len(train_cache),
                          'train_cached_ids': sum(len(s) + len(c) for s, c in train_cache)}

def online_batch(indices, destination=None):
    destination = device if destination is None else destination
    rows = [train[i] for i in indices]
    src = pad_sequence([torch.tensor([BOS] + encode(r['question']) + [EOS], dtype=torch.long) for r in rows],
                       batch_first=True, padding_value=PAD).to(destination)
    codes = [encode(r['code']) for r in rows]
    tgt = pad_sequence([torch.tensor([BOS] + c, dtype=torch.long) for c in codes],
                       batch_first=True, padding_value=PAD).to(destination)
    labels = pad_sequence([torch.tensor(c + [EOS], dtype=torch.long) for c in codes],
                          batch_first=True, padding_value=PAD).to(destination)
    return src, tgt, labels

def cached_batch(indices, destination=None, cache_rows=None):
    destination = device if destination is None else destination
    cache_rows = train_cache if cache_rows is None else cache_rows
    src = pad_sequence([torch.tensor(cache_rows[i][0], dtype=torch.long) for i in indices],
                       batch_first=True, padding_value=PAD).to(destination)
    codes = [cache_rows[i][1] for i in indices]
    tgt = pad_sequence([torch.tensor([BOS] + c, dtype=torch.long) for c in codes],
                       batch_first=True, padding_value=PAD).to(destination)
    labels = pad_sequence([torch.tensor(c + [EOS], dtype=torch.long) for c in codes],
                          batch_first=True, padding_value=PAD).to(destination)
    return src, tgt, labels

plans = []
for repeat in range(REPEATS):
    batch_rng = random.Random(SEED + 1000 * repeat)
    plans.append([batch_rng.sample(range(len(train)), BATCH_SIZE)
                  for _ in range(BENCH_WARMUP + MEASURE_STEPS + PROFILE_STEPS)])
parity_batches = plans[0][:16] + [[0, len(train) - 1,
    max(range(len(train_cache)), key=lambda i: len(train_cache[i][0])),
    max(range(len(train_cache)), key=lambda i: len(train_cache[i][1]))]]
for indices in parity_batches:
    a, b = online_batch(indices, 'cpu'), cached_batch(indices, 'cpu')
    assert all(torch.equal(x, y) for x, y in zip(a, b)), 'Cache changed batch inputs/labels'
    assert torch.equal(a[1].ne(PAD), a[2].ne(PAD))
result['checks'] = {'online_cached_batches_exactly_equal': len(parity_batches),
                     'paired_batch_plans_sha256': data_hash(plans)}
save_results()
print('CACHE', json.dumps(result['tokenization']))
print('BATCH PARITY', len(parity_batches))


## 4. Та же модель и одинаковое начальное состояние
Ни ширину, ни число слоёв, ни dropout не меняем. Перед каждым режимом восстанавливаем исходные веса и RNG; состояния AdamW начинаются пустыми, как при обучении с нуля.


In [ ]:
class SwiGLUInput(nn.Module):
    """Two input-dependent branches: SiLU(gate) * value, before FFN dropout."""
    def __init__(self, width, hidden):
        super().__init__()
        self.gate = nn.Linear(width, hidden)
        self.value = nn.Linear(width, hidden)
    def forward(self, x):
        return F.silu(self.gate(x)) * self.value(x)

def replace_ffn_with_swiglu(transformer):
    # Build the original model first: embeddings/attention/norms/head keep seed42 initialization.
    # Isolated CPU RNG initializes only the new FFNs; training RNG is restored on exit.
    with torch.random.fork_rng(devices=[]):
        torch.set_rng_state(torch.Generator().manual_seed(SEED).get_state())
        for layer in list(transformer.encoder.layers) + list(transformer.decoder.layers):
            layer.linear1 = SwiGLUInput(D_MODEL, SWIGLU_DIM)
            layer.linear2 = nn.Linear(SWIGLU_DIM, D_MODEL)
            layer.activation = nn.Identity()  # SiLU and multiplication already happen in linear1.
            if isinstance(layer, nn.TransformerEncoderLayer):
                # Fused encoder supports only ReLU/GELU and would bypass the gated module.
                layer.activation_relu_or_gelu = 0
            for module in [layer.linear1.gate, layer.linear1.value, layer.linear2]:
                nn.init.xavier_uniform_(module.weight)

class SimpleTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.tokens = nn.Embedding(VOCAB, D_MODEL, padding_idx=PAD)
        self.positions = nn.Embedding(MAX_POSITIONS, D_MODEL)
        self.transformer = nn.Transformer(d_model=D_MODEL, nhead=4,
            num_encoder_layers=N_LAYERS, num_decoder_layers=N_LAYERS,
            dim_feedforward=4 * D_MODEL, dropout=0.1, batch_first=True, norm_first=True)
        self.transformer.encoder.enable_nested_tensor = False
        self.transformer.encoder.use_nested_tensor = False
        self.head = nn.Linear(D_MODEL, VOCAB)
        replace_ffn_with_swiglu(self.transformer)
    def embed(self, ids):
        if ids.shape[1] > MAX_POSITIONS:
            raise ValueError('Sequence too long; no silent truncation')
        pos = torch.arange(ids.shape[1], device=ids.device)
        return self.tokens(ids) * math.sqrt(D_MODEL) + self.positions(pos)
    def decode_prefix(self, tgt, memory, source_padding):
        mask = torch.ones(tgt.shape[1], tgt.shape[1], device=tgt.device, dtype=torch.bool).triu(1)
        return self.transformer.decoder(self.embed(tgt), memory, tgt_mask=mask,
            tgt_key_padding_mask=tgt.eq(PAD), memory_key_padding_mask=source_padding)
    def forward(self, src, tgt):
        memory = self.transformer.encoder(self.embed(src), src_key_padding_mask=src.eq(PAD))
        return self.head(self.decode_prefix(tgt, memory, src.eq(PAD)))
    @torch.inference_mode()
    def generate(self, src, max_new_tokens=512):
        self.eval()
        memory = self.transformer.encoder(self.embed(src), src_key_padding_mask=src.eq(PAD))
        tgt = torch.full((len(src), 1), BOS, device=src.device, dtype=torch.long)
        done = torch.zeros(len(src), device=src.device, dtype=torch.bool)
        for _ in range(max_new_tokens):
            hidden = self.decode_prefix(tgt, memory, src.eq(PAD))
            logits = self.head(hidden[:, -1])
            logits[:, [PAD, BOS]] = -torch.inf
            token = logits.argmax(-1).masked_fill(done, PAD)
            done |= token.eq(EOS)
            tgt = torch.cat([tgt, token[:, None]], dim=1)
            if bool(done.all()):
                break
        return tgt[:, 1:]

def learning_rate_for_update(update, warmup_steps=WARMUP_STEPS, total_steps=SCHEDULE_TOTAL_STEPS,
                             peak_lr=LEARNING_RATE, min_lr=MIN_LEARNING_RATE):
    # Index1 is the first update. Clamp after training to prevent a cosine restart.
    update = min(max(int(update), 1), total_steps)
    if update <= warmup_steps:
        return peak_lr * update / warmup_steps
    progress = (update - warmup_steps) / (total_steps - warmup_steps)
    return min_lr + 0.5 * (peak_lr - min_lr) * (1 + math.cos(math.pi * progress))

def learning_rate_multiplier(completed_updates):
    return learning_rate_for_update(completed_updates + 1) / LEARNING_RATE

torch.manual_seed(SEED)
with warnings.catch_warnings():
    warnings.filterwarnings('ignore', message='enable_nested_tensor is True.*')
    initial_model = SimpleTransformer()
initial_weights = {k: v.detach().cpu().clone() for k, v in initial_model.state_dict().items()}
initial_cpu_rng = torch.get_rng_state().clone()
parameters = sum(p.numel() for p in initial_model.parameters())
assert parameters == 6306112
assert all(v.dtype == torch.float32 for v in initial_weights.values())
del initial_model

def state_hash(weights):
    digest = hashlib.sha256()
    for name, value in sorted(weights.items()):
        digest.update(name.encode())
        digest.update(value.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()

initial_weights_hash = state_hash(initial_weights)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)

def new_training_state(use_amp, repeat):
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', message='enable_nested_tensor is True.*')
        model = SimpleTransformer().to(device)
    model.load_state_dict(initial_weights)
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=learning_rate_multiplier)
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
    counts = {'updates': 0}
    def after_step(optimizer, args, kwargs):
        counts['updates'] += 1
    optimizer.register_step_post_hook(after_step)
    assert not optimizer.state
    assert state_hash(model.state_dict()) == initial_weights_hash
    torch.set_rng_state(initial_cpu_rng)
    random.seed(SEED + 1000 * repeat)
    if device.type == 'cuda':
        torch.cuda.manual_seed_all(SEED + 1000 * repeat)
    return {'model': model, 'optimizer': optimizer, 'scheduler': scheduler,
            'scaler': scaler, 'counts': counts, 'amp': use_amp,
            'initial_cpu_rng_sha256': sha(torch.get_rng_state().numpy().tobytes()),
            'initial_device_rng_sha256': sha((torch.cuda.get_rng_state(device) if device.type == 'cuda'
                                              else torch.get_rng_state()).numpy().tobytes())}

def update_step(state, batch):
    model, optimizer, scaler = state['model'], state['optimizer'], state['scaler']
    src, tgt, labels = batch
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=state['amp']):
        logits = model(src, tgt)
        loss = criterion(logits.reshape(-1, VOCAB), labels.reshape(-1))
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)  # disabled scaler: no-op; clipping always sees unscaled gradients
    norm = nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    before = state['counts']['updates']
    scaler.step(optimizer)
    scaler.update()
    successful = state['counts']['updates'] > before
    if successful:
        state['scheduler'].step()
    result['total_attempted_updates'] += 1
    result['total_successful_updates'] += int(successful)
    return loss.detach(), norm.detach(), successful

result['model'] = {'parameters': parameters, 'd_model': D_MODEL, 'encoder_layers': N_LAYERS,
                   'decoder_layers': N_LAYERS, 'heads': 4, 'swiglu_dim': SWIGLU_DIM,
                   'norm_first': True, 'initial_weights_sha256': initial_weights_hash,
                   'source': 'same model definitions and initialization as prepared15',
                   'weights_dtype': 'float32', 'optimizer_moments_dtype': 'float32'}
save_results()
print('MODEL', json.dumps(result['model']))


## 5. Таймер и отдельный профиль

Основное время — весь train-блок с CUDA sync только на границах. Loss/norm переносятся на CPU после остановки таймера; внутри нет print, JSON или дополнительной проверки каждого шага через `.item()`.

В отдельном **сериализованном** профиле считаем CPU-подготовку и CUDA event spans для переноса и вычислений. Он меняет перекрытие CPU/GPU и добавляет синхронизации, поэтому его времена не складываем с основным замером. Compute span может включать ожидание CPU/запуска kernels; это не GPU utilization и не сумма чистого kernel time. Пик памяти относится именно к обучению.


In [ ]:
def finite_parameters(model):
    return bool(torch.stack([torch.isfinite(p).all() for p in model.parameters()]).all().item())

def trace_stats(values):
    numbers = torch.stack(values).float().cpu().tolist()
    finite = [x for x in numbers if math.isfinite(x)]
    return {'nonfinite': len(numbers) - len(finite),
            'first5_mean': statistics.mean(finite[:5]) if finite else None,
            'last5_mean': statistics.mean(finite[-5:]) if finite else None,
            'min': min(finite) if finite else None, 'max': max(finite) if finite else None}

def run_phase(state, build_batch, batches):
    useful_tokens = sum(len(train_cache[i][1]) + 1 for indices in batches for i in indices)
    padded_tokens = sum((max(len(train_cache[i][1]) for i in indices) + 1) * len(indices) for indices in batches)
    losses, norms = [], []
    before = state['counts']['updates']
    lr_start = state['optimizer'].param_groups[0]['lr']
    scale_start = state['scaler'].get_scale()
    sync()
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)
    started = time.perf_counter()
    for indices in batches:
        loss, norm, _ = update_step(state, build_batch(indices))
        losses.append(loss)
        norms.append(norm)
    sync()
    seconds = time.perf_counter() - started
    successful = state['counts']['updates'] - before
    report = {'attempts': len(batches), 'successful_updates': successful,
              'skipped_updates': len(batches) - successful, 'seconds': seconds,
              'ms_per_attempt': seconds * 1000 / len(batches),
              'successful_updates_per_second': successful / seconds,
              'useful_target_tokens': useful_tokens, 'useful_target_tokens_per_second': useful_tokens / seconds,
              'target_padding_fraction': 1 - useful_tokens / padded_tokens,
              'lr_start': lr_start, 'lr_next': state['optimizer'].param_groups[0]['lr'],
              'scale_start': scale_start, 'scale_end': state['scaler'].get_scale(),
              'loss': trace_stats(losses), 'unscaled_gradient_norm': trace_stats(norms),
              'peak_training_allocated_mib': torch.cuda.max_memory_allocated(device) / 2**20 if device.type == 'cuda' else None,
              'peak_training_reserved_mib': torch.cuda.max_memory_reserved(device) / 2**20 if device.type == 'cuda' else None}
    report['parameters_finite'] = finite_parameters(state['model'])
    return report

@torch.inference_mode()
def validation_loss_fp32(model):
    model.eval()
    total, tokens = 0.0, 0
    try:
        for start in range(0, len(validation_cache), BATCH_SIZE):
            src, tgt, labels = cached_batch(range(start, min(start + BATCH_SIZE, len(validation_cache))), cache_rows=validation_cache)
            logits = model(src, tgt)
            total += F.cross_entropy(logits.reshape(-1, VOCAB), labels.reshape(-1), ignore_index=PAD, reduction='sum').item()
            tokens += labels.ne(PAD).sum().item()
        value = total / tokens
        return value if math.isfinite(value) else None
    finally:
        model.train()

def serial_profile(state, build_batch, batches):
    rows = []
    for indices in batches:
        sync()
        cpu_started = time.perf_counter()
        cpu_batch = build_batch(indices, destination='cpu')
        cpu_ms = (time.perf_counter() - cpu_started) * 1000
        if device.type == 'cuda':
            transfer_start, transfer_end, compute_start, compute_end = [torch.cuda.Event(enable_timing=True) for _ in range(4)]
            transfer_start.record()
            batch = tuple(t.to(device) for t in cpu_batch)
            transfer_end.record()
            compute_start.record()
            loss, norm, successful = update_step(state, batch)
            compute_end.record()
            sync()
            transfer_ms = transfer_start.elapsed_time(transfer_end)
            compute_ms = compute_start.elapsed_time(compute_end)
        else:
            transfer_ms = None
            started = time.perf_counter()
            loss, norm, successful = update_step(state, cpu_batch)
            compute_ms = (time.perf_counter() - started) * 1000
        rows.append({'cpu_prepare_ms': cpu_ms, 'transfer_stream_span_ms': transfer_ms,
                     'compute_stream_span_ms': compute_ms, 'successful': bool(successful),
                     'finite_loss': bool(torch.isfinite(loss).item()),
                     'finite_unscaled_gradient_norm': bool(torch.isfinite(norm).item())})
    return {'samples': len(rows), 'cpu_prepare_ms_median': statistics.median(r['cpu_prepare_ms'] for r in rows),
            'transfer_stream_span_ms_median': statistics.median(r['transfer_stream_span_ms'] for r in rows) if device.type == 'cuda' else None,
            'compute_stream_span_ms_median': statistics.median(r['compute_stream_span_ms'] for r in rows),
            'skipped_updates': sum(not r['successful'] for r in rows),
            'finite_losses_and_norms': all(r['finite_loss'] and r['finite_unscaled_gradient_norm'] for r in rows),
            'scope': 'separate serialized profile, not additive decomposition of primary block; event spans include launch/host waits',
            'rows': rows}


## 6. Запустить сравнение
Каждый законченный режим сразу сохраняется. Короткие блоки не оценивают итоговую правильность программ.


In [ ]:
assert not result['benchmark_started'], 'Benchmark уже запускался; скачай результаты и начни новую сессию'
result['benchmark_started'] = True
result['phase'] = 'benchmark_running'
save_results()
benchmark_started = time.perf_counter()
try:
    for repeat in range(REPEATS):
        order = MODES[repeat % len(MODES):] + MODES[:repeat % len(MODES)]
        fp32_ends = {}
        rng_starts = []
        for mode in order:
            print(f'Repeat {repeat + 1}/{REPEATS}: {mode}', flush=True)
            gc.collect()
            if device.type == 'cuda':
                torch.cuda.empty_cache()
            state = new_training_state(mode.startswith('C_'), repeat)
            rng_starts.append((state['initial_cpu_rng_sha256'], state['initial_device_rng_sha256']))
            build_batch = online_batch if mode.startswith('A_') else cached_batch
            plan = plans[repeat]
            warmup = run_phase(state, build_batch, plan[:BENCH_WARMUP])
            measured = run_phase(state, build_batch, plan[BENCH_WARMUP:BENCH_WARMUP + MEASURE_STEPS])
            assert all(p.dtype == torch.float32 for p in state['model'].parameters())
            assert all(v.dtype == torch.float32 for values in state['optimizer'].state.values()
                       for key, v in values.items() if key in ('exp_avg', 'exp_avg_sq'))
            row = {'repeat': repeat + 1, 'mode': mode, 'warmup': warmup, 'measured': measured,
                   'validation64_loss_fp32': validation_loss_fp32(state['model']),
                   'successful_updates_total_before_profile': state['counts']['updates'],
                   'scheduler_completed_updates': state['scheduler'].last_epoch,
                   'scaler_state': state['scaler'].state_dict()}
            row['stable_measured_block'] = (measured['skipped_updates'] == 0 and measured['loss']['nonfinite'] == 0
                and measured['unscaled_gradient_norm']['nonfinite'] == 0 and measured['parameters_finite'])
            assert row['scheduler_completed_updates'] == row['successful_updates_total_before_profile']
            result['runs'].append(row)
            if not mode.startswith('C_'):
                fp32_ends[mode] = {k: v.detach().cpu().clone() for k, v in state['model'].state_dict().items()}
            save_results()
            print(f"  {measured['ms_per_attempt']:.3f} ms/attempt; "
                  f"{measured['useful_target_tokens_per_second']:.0f} useful target tokens/s; "
                  f"skips={measured['skipped_updates']}; stable={row['stable_measured_block']}", flush=True)
            if repeat == 0:
                profile = serial_profile(state, build_batch, plan[-PROFILE_STEPS:])
                result['profiles'].append({'mode': mode, **profile})
                save_results()
            del state
            gc.collect()
        assert len(set(rng_starts)) == 1, 'Initial RNG differs between paired modes'
        a, b = fp32_ends[MODES[0]], fp32_ends[MODES[1]]
        max_error = max((a[k] - b[k]).abs().max().item() for k in a)
        result['fp32_pair_checks'].append({'repeat': repeat + 1, 'max_abs_final_weight_difference': max_error,
                                          'initial_RNG_equal': True,
                                          'scope': 'FP32 online/cache trajectories; small CUDA nondeterminism possible'})
        del fp32_ends, a, b
        save_results()
    result['benchmark_finished'] = True
    result['phase'] = 'benchmark_complete'
except BaseException as error:
    result['phase'] = 'interrupted_or_failed'
    result['error_type'] = type(error).__name__
    raise
finally:
    result['benchmark_wall_seconds_including_warmup_profile_evaluation'] = time.perf_counter() - benchmark_started
    save_results()


## 7. Сравнить результаты

Смотрим медиану и разброс трёх повторов. A→B — выигрыш кеша, B→C — AMP. Скромную разницу в пределах разброса не считать устойчивым ускорением. Cache setup имеет разовую цену; break-even ниже оценивается по разнице времени шага.

Для выбора AMP нужны конечные loss/градиенты/веса и отсутствие skips в измеряемом блоке. Даже при хорошей скорости следующим отдельным запуском нужен контроль качества50k. Текущий короткий validation loss не заменяет его.


In [ ]:
summary_rows = []
for mode in MODES:
    rows = [r for r in result['runs'] if r['mode'] == mode]
    if not rows:
        continue
    times = [r['measured']['ms_per_attempt'] for r in rows]
    memory = [r['measured']['peak_training_allocated_mib'] for r in rows]
    summary_rows.append({'mode': mode, 'completed_repeats': len(rows),
        'ms_median': statistics.median(times), 'ms_min': min(times), 'ms_max': max(times),
        'useful_target_tokens_per_second_median': statistics.median(r['measured']['useful_target_tokens_per_second'] for r in rows),
        'training_peak_allocated_mib_max': max(memory) if all(x is not None for x in memory) else None,
        'measured_skips_total': sum(r['measured']['skipped_updates'] for r in rows),
        'all_measured_blocks_stable': all(r['stable_measured_block'] for r in rows)})
result['summary'] = summary_rows
paired = []
for repeat in range(1, REPEATS + 1):
    by_mode = {r['mode']: r['measured'] for r in result['runs'] if r['repeat'] == repeat}
    if all(m in by_mode for m in MODES):
        a, b, c = (by_mode[m]['seconds'] for m in MODES)
        paired.append({'repeat': repeat, 'cache_speedup_A_over_B': a / b,
                       'AMP_speedup_B_over_C': b / c, 'combined_speedup_A_over_C': a / c})
result['paired_speedups'] = paired
by_mode = {r['mode']: r for r in summary_rows}
if all(m in by_mode for m in MODES):
    delta_seconds = (by_mode[MODES[0]]['ms_median'] - by_mode[MODES[1]]['ms_median']) / 1000
    result['cache_break_even_updates_estimate'] = cache_seconds / delta_seconds if delta_seconds > 0 else None
    result['cache_break_even_scope'] = 'cache setup seconds divided by median A-B step saving; excludes download, approximate'
result['phase'] = 'summary_ready' if result['benchmark_finished'] else 'partial_summary'
save_results()
print('MODE                         median ms    min ms    max ms   target tokens/s   skips   stable')
for r in summary_rows:
    print(f"{r['mode']:<28} {r['ms_median']:9.3f} {r['ms_min']:9.3f} {r['ms_max']:9.3f} "
          f"{r['useful_target_tokens_per_second_median']:15.0f} {r['measured_skips_total']:7} "
          f"{r['all_measured_blocks_stable']}")
print('PAIRED SPEEDUPS', json.dumps(paired))
print('SERIAL PROFILES', json.dumps([{k: v for k, v in p.items() if k != 'rows'} for p in result['profiles']]))
print('CACHE BREAK-EVEN UPDATES', result.get('cache_break_even_updates_estimate'))
print('FP32 PAIR CHECKS', json.dumps(result['fp32_pair_checks']))
with (RUN_DIR / 'summary.csv').open('w', encoding='utf-8', newline='') as f:
    if summary_rows:
        writer = csv.DictWriter(f, fieldnames=list(summary_rows[0]))
        writer.writeheader()
        writer.writerows(summary_rows)


## 8. Скачать ZIP и IPYNB с outputs

Эта ячейка ничего не обучает. Её можно выполнить после прерывания benchmark: JSON уже содержит законченные режимы и счётчики. Веса коротких замеров не сохраняются и не используются как новая качественная модель. В ZIP — результаты/CSV/manifest; отдельно скачай этот notebook с outputs.


In [ ]:
assert RESULTS_FILE.exists(), 'Сначала выполни setup'
files_to_save = [RESULTS_FILE]
if (RUN_DIR / 'summary.csv').exists():
    files_to_save.append(RUN_DIR / 'summary.csv')
manifest = [{'name': path.name, 'bytes': path.stat().st_size, 'sha256': sha(path.read_bytes())}
            for path in files_to_save]
with zipfile.ZipFile(ARCHIVE_FILE, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in files_to_save:
        archive.write(path, path.name)
    archive.writestr('manifest.json', json.dumps(manifest, ensure_ascii=False, indent=2))
with zipfile.ZipFile(ARCHIVE_FILE) as archive:
    assert archive.testzip() is None
print('ARCHIVE', ARCHIVE_FILE.name, ARCHIVE_FILE.stat().st_size, 'bytes')
try:
    from google.colab import files
except ImportError:
    print('Скачай ZIP из файловой панели; отдельно сохрани IPYNB с outputs.')
else:
    files.download(str(ARCHIVE_FILE))


## Источники

- Подготовленный notebook15 и сохранённые результаты15: неизменные модель, данные, BPE, LR и batch8. Текущий benchmark начинает с исходных весов, не с checkpoint100k.
- [PyTorch2.11 AMP examples](https://docs.pytorch.org/docs/2.11/notes/amp_examples.html): autocast, GradScaler, unscale перед clipping; документация прочитана при подготовке.
- [PyTorch2.11 CUDA Event](https://docs.pytorch.org/docs/2.11/generated/torch.cuda.Event.html), [optimizer post-hook](https://docs.pytorch.org/docs/2.11/generated/torch.optim.Optimizer.register_step_post_hook.html): замер stream span и подсчёт выполненных optimizer steps.
- [NVIDIA T4](https://www.nvidia.com/en-us/data-center/tesla-t4/): FP16 Tensor Cores. Выигрыш на нашей модели ещё не измерен.

Выполнение на CPU при подготовке проверяет код, а не скорость T4. Новые результаты T4 появятся только после пользовательского запуска. План/гипотеза — наш выбор, не рекомендация источников для именно этой модели.
